In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.core.configuration import Configuration
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [2]:
M_xanthus = read_sbml_model("../M_xanthus_model_V3.xml")
M_xanthus

Name,myxo_model
Memory address,7a6fc1e960c0
Number of metabolites,1223
Number of reactions,1337
Number of genes,1200
Number of groups,0
Objective expression,1.0*OF_BIOMASS - 1.0*OF_BIOMASS_reverse_80d2e
Compartments,"c, e"


In [554]:
M_xanthus.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
Fe3_e,EX_Fe3_e,1.061,0,0.00%
ac_e,EX_ac_e,119.4,2,1.82%
alaala_e,EX_alaala_e,507.1,6,23.20%
ca2_e,EX_ca2_e,0.3535,0,0.00%
cl_e,EX_cl_e,0.3535,0,0.00%
cobalt2_e,EX_cobalt2_e,0.3535,0,0.00%
cu2_e,EX_cu2_e,0.3535,0,0.00%
glu_L_e,EX_glu_L_e,41.29,5,1.57%
his_L_e,EX_his_L_e,16.9,6,0.77%
ile_L_e,EX_ile_L_e,51.56,6,2.36%


In [ ]:
iMAT_res = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/results/quantiles/iMAT/log2FoldChange/epsilon_1.0_quantiles_40_70_name.csv", sep=";", index_col="Unnamed: 0")
#iMAT_res = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/results/iMat/reactionData_classification_name.csv", sep=";", index_col="Unnamed: 0")
iMAT_res

,reaction_id,flux_value,classification,y_f,y_r
0,"2-amino-4-hydroxy-6-hydroxymethyl-7,8-dihydrop...",0.0,moderate,NaN,NaN
1,gamma-L-glutamyl-L-cysteine:glycine ligase (AD...,1.0,high,1.0,0.0
2,R07600 [c],0.0,low,1.0,0.0
3,IMP:diphosphate phospho-D-ribosyltransferase [c],0.0,moderate,NaN,NaN
4,"2,3-dihydro-2,3-dihydroxybenzoate:NAD+ oxidore...",0.0,low,1.0,0.0
...,...,...,...,...,...
1362,Exchange for D-Lactate [e],255.0,NaN,NaN,NaN
1363,Exchange for H2S2O3 [e],0.0,NaN,NaN,NaN
1364,Demand for glycogen(n-1) [c],0.0,NaN,NaN,NaN
1365,Demand for Biomass [c],0.0,NaN,NaN,NaN


In [ ]:
iMAT_res_filter = iMAT_res[iMAT_res.flux_value != 0] # take all the flux different from 0 (so active one)
#iMAT_res_filter = iMAT_res[iMAT_res.flux != 0]
iMAT_res_filter

,reaction_id,flux_value,classification,y_f,y_r
1,gamma-L-glutamyl-L-cysteine:glycine ligase (AD...,1.000000,high,1.0,0.0
12,L-glutamate:L-cysteine gamma-ligase (ADP-formi...,1.000000,high,1.0,0.0
14,ATP:dCDP phosphotransferase [c],1.000000,high,1.0,0.0
26,oxalosuccinate carboxy-lyase (2-oxoglutarate-f...,501.142860,moderate,NaN,NaN
31,Sulfate adenyltransferase [c],1.000000,high,1.0,0.0
...,...,...,...,...,...
1341,Exchange for Enterobactin [e],1000.000000,NaN,NaN,NaN
1353,Exchange for NH3 [e],5.785714,NaN,NaN,NaN
1357,Exchange for Acetoacetate [e],-754.214290,NaN,NaN,NaN
1360,Exchange for O2 [e],-870.500000,NaN,NaN,NaN


In [557]:
# Create dictionary to convert name to id
Dico_reaction = {}
for i in M_xanthus.reactions._dict:
    Dico_reaction[M_xanthus.reactions.get_by_id(i).name] = i
print(Dico_reaction)

# create a list of reaction id that should have flux
active_list = []
for i in iMAT_res_filter["reaction_id"]:
    active_list.append(Dico_reaction[i])
print(active_list)

{'2-amino-4-hydroxy-6-hydroxymethyl-7,8-dihydropteridine-diphosphate:4-aminobenzoate 2-amino-4-hydroxydihydropteridine-6-methenyltransferase [c]': 'rxn02201_c', 'gamma-L-glutamyl-L-cysteine:glycine ligase (ADP-forming) [c]': 'rxn00351_c', 'R07600 [c]': 'rxn07431_c', 'IMP:diphosphate phospho-D-ribosyltransferase [c]': 'rxn00836_c', '2,3-dihydro-2,3-dihydroxybenzoate:NAD+ oxidoreductase [c]': 'rxn01094_c', 'acetyl-CoA:L-serine O-acetyltransferase [c]': 'rxn00423_c', 'palmitoyl-lipoteichoic acid synthesis (n=24), linked, glucose substituted [c]': 'rxn10298_c', 'ATP:CMP phosphotransferase [c]': 'rxn00364_c', 'Transport of dicarboxylates, extracellular [c]': 'rxn05561_c', 'UDP-N-acetyl-D-glucosamine:undecaprenyl-diphospho-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-meso-2,6-diaminopimeloyl-D-alanyl-D-alanine 4-beta-N-acetylglucosaminlytransferase [c]': 'rxn03408_c', 'Isochorismate pyruvate-hydrolase [c]': 'rxn02177_c', 'FACOAL160(ISO) [c]': 'rxn05250_c', 'L-glutamate:L-cysteine gamma-ligase 

Check reaction bounds and change the bounds (0.0, 0.0) looking at ModelSeed

In [558]:
for i in active_list:
    print(M_xanthus.reactions.get_by_id(i).id + ": " + str(M_xanthus.reactions.get_by_id(i).bounds))

rxn00351_c: (-1000.0, 1000.0)
rxn00646_c: (0.0, 1000.0)
rxn01673_c: (-1000.0, 1000.0)
rxn00199_c: (0.0, 1000.0)
rxn09240_c: (0.0, 1000.0)
rxn00172_c: (-1000.0, 0.0)
rxn02168_c: (-1000.0, 1000.0)
rxn00192_c: (0.0, 1000.0)
rxn00262_c: (-1000.0, 1000.0)
rxn10126_c: (0.0, 1000.0)
rxn15230_c: (-1000.0, 1000.0)
rxn04792_c: (-1000.0, 1000.0)
rxn11268_c: (-1000.0, 0.0)
rxn00910_c: (-1000.0, 1000.0)
rxn00717_c: (-1000.0, 1000.0)
rxn00416_c: (-1000.0, 1000.0)
rxn02376_c: (-1000.0, 1000.0)
rxn00361_c: (0.0, 1000.0)
rxn08971_c: (0.0, 1000.0)
rxn05569_c: (0.0, 0.0)
rxn08976_c: (0.0, 1000.0)
rxn00781_c: (-1000.0, 1000.0)
rxn05298_c: (-1000.0, 1000.0)
rxn01451_c: (-1000.0, 1000.0)
rxn00097_c: (-1000.0, 1000.0)
rxn00802_c: (-1000.0, 1000.0)
rxn00178_c: (-1000.0, 1000.0)
rxn00549_c: (0.0, 1000.0)
rxn00305_c: (-1000.0, 1000.0)
rxn00269_c: (-1000.0, 1000.0)
rxn05466_c: (-1000.0, 1000.0)
rxn00395_c: (0.0, 1000.0)
rxn00471_c: (0.0, 1000.0)
rxn05595_c: (-1000.0, 1000.0)
rxn09188_c: (-1000.0, 1000.0)
rxn0033

For ModelSeed:

rxn08350 rxn05469 rxn05683 rxn05484 rxn12844 rxn05569

In [559]:
M_xanthus.reactions.rxn05469_c.bounds = [-1000, 1000]
M_xanthus.reactions.rxn05469_c

Reaction identifier,rxn05469_c
Name,pyruvate reversible transport via proton symport [c]
Memory address,0x7aa041333920
Stoichiometry,h_e + pyr_e <=> h_c + pyr_c H+ [e] + Pyruvate [e] <=> H+ [c] + Pyruvate [c]
GPR,
Lower bound,-1000
Upper bound,1000


In [560]:
M_xanthus.reactions.rxn05484_c.bounds = [-1000, 1000]
M_xanthus.reactions.rxn05484_c

Reaction identifier,rxn05484_c
Name,acetoacetate transport via proton symport [c]
Memory address,0x7aa0418beae0
Stoichiometry,acac_e + h_e <=> acac_c + h_c Acetoacetate [e] + H+ [e] <=> Acetoacetate [c] + H+ [c]
GPR,MXAN_0704
Lower bound,-1000
Upper bound,1000


In [561]:
M_xanthus.reactions.rxn05569_c.bounds = [0, 1000]
M_xanthus.reactions.rxn05569_c

Reaction identifier,rxn05569_c
Name,D-glucosamine transport via PEP:Pyr PTS [c]
Memory address,0x7aa041e7f2c0
Stoichiometry,gam_e + pep_c --> gam6p_c + pyr_c GLUM [e] + Phosphoenolpyruvate [c] --> D-Glucosamine phosphate [c] + Pyruvate [c]
GPR,MXAN_6530
Lower bound,0
Upper bound,1000


In [562]:
M_xanthus.reactions.rxn05683_c.bounds = [-1000,1000]
M_xanthus.reactions.rxn05683_c

Reaction identifier,rxn05683_c
Name,"Butyrate transport via proton symport, reversible [c]"
Memory address,0x7aa0418bfe90
Stoichiometry,but_e + h_e <=> but_c + h_c Butyrate [e] + H+ [e] <=> Butyrate [c] + H+ [c]
GPR,MXAN_0704
Lower bound,-1000
Upper bound,1000


In [563]:
M_xanthus.reactions.rxn08350_c.bounds = [0, 1000]
M_xanthus.reactions.rxn08350_c

Reaction identifier,rxn08350_c
Name,D-lactate transport via proton symport (periplasm) [c]
Memory address,0x7aa0413715b0
Stoichiometry,h_e + lac_D_e --> h_c + lac_D_c H+ [e] + D-Lactate [e] --> H+ [c] + D-Lactate [c]
GPR,
Lower bound,0
Upper bound,1000


In [564]:
M_xanthus.reactions.rxn12844_c.bounds = [-1000,1000]
M_xanthus.reactions.rxn12844_c

Reaction identifier,rxn12844_c
Name,Gly-Cys aminopeptidase [c]
Memory address,0x7aa041df6750
Stoichiometry,gly_cys_L_c + h2o_c <=> cys_L_c + gly_c Gly-Cys [c] + H2O [c] <=> L-Cysteine [c] + Glycine [c]
GPR,(MXAN_6714 and MXAN_7084) or MXAN_4963 or MXAN_6822 or MXAN_1160 or MXAN_1160
Lower bound,-1000
Upper bound,1000


Force to activate specific reaction

In [565]:
M_xanthus.reactions.rxn00910_c

Reaction identifier,rxn00910_c
Name,5-methyltetrahydrofolate:NADP+ oxidoreductase [c]
Memory address,0x7aa041ecf560
Stoichiometry,5mthf_c + nadp_c <=> h_c + mlthf_c + nadph_c 5-Methyltetrahydrofolate [c] + NADP [c] <=> H+ [c] + 5-10-Methylenetetrahydrofolate [c] + NADPH [c]
GPR,MXAN_3039
Lower bound,-1000.0
Upper bound,1000.0


In [566]:
FBA = M_xanthus.optimize()
FBA.fluxes["rxn00910_c"]

np.float64(0.0)

In [ ]:
# Add constraint that force the fluxe to pass there / Infeasable
y = M_xanthus.problem.Variable("y_RXN", type="binary") # add a binary constraint
M_xanthus.add_cons_vars(y)

for i in active_list:
    have_flux = M_xanthus.problem.Constraint(
        M_xanthus.reactions.get_by_id(i).flux_expression - 1 + 1000*(1 - y), # - 1 [threshold] + 1000 [big number] * (1 - y) [if 1 = big number] [if 0 = - big number]
        lb=0,
        ub=0) # force the reaction to have at least 1 (or -1?) flux
    M_xanthus.add_cons_vars(have_flux)

M_xanthus.solver.update()

In [568]:
M_xanthus.reactions.get_by_id("rxn00910_c").bounds

(-1000.0, 1000.0)

In [569]:
FBA = M_xanthus.optimize()
FBA.fluxes["rxn00910_c"]

/home/mickael/miniconda3/envs/Micka_Predation/lib/python3.12/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


np.float64(0.0)

In [570]:
M_xanthus.slim_optimize()

nan

Shut down all others reactions

In [ ]:
for i in M_xanthus.reactions: # Shut down all the non active reaction
    if i.id not in active_list:
        i.bounds = [0,0]
M_xanthus.reactions.OF_BIOMASS.bounds = [-1000, 1000] # except Biomass

M_xanthus.slim_optimize() # Didn't growth

nan

New Data

python3 IntegrationPackage/main.py weighted_iMAT -f 'data/Raw/WT_vs_4preys.txt' -g 'GeneNames' -i 'W1' -m 'M_xanthus_model_V3_hdca_iMAT.xml' -o results/ -d "quantile"

In [6]:
DataTable = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys.txt", sep = "\t")
DataTable

,GeneNames,Ec1,Ec2,Ec3,B1,B2,B3,C1,C2,C3,...,W3,K1,K2,K3,WC1,WC2,WC3,KC1,KC2,KC3
0,WP_002614080.1,13032,13136,12917,18050,22828,16816,42888,32404,20170,...,60053,35638,38908,37314,15344,15530,27018,17185,13886,9121
1,WP_002614803.1,6817,5982,6246,15018,16718,18070,29227,34305,9713,...,69546,28536,31303,34021,19022,19870,35902,12388,22264,17843
2,WP_002633201.1,275,195,239,521,550,406,1140,727,386,...,2409,838,832,531,1110,995,1063,1841,1027,766
3,WP_002633598.1,15516,14792,13306,28206,30917,22916,64389,58243,23106,...,62136,27039,28864,34478,7962,9148,24438,4169,7524,7193
4,WP_002633601.1,4095,2680,1969,8170,8806,14771,20987,18394,9361,...,17038,13309,11109,20035,3338,2187,11373,3164,2406,4588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7184,WP_141276995.1,31,36,108,26,31,49,93,69,45,...,76,29,39,42,29,26,39,23,19,33
7185,WP_141277062.1,11,13,46,11,17,4,43,46,17,...,79,10,15,13,14,33,26,17,19,31
7186,WP_143049088.1,1332,1512,1834,2258,2340,1791,4704,5134,1659,...,13345,3235,3431,2941,4105,5714,3841,9265,5144,6080
7187,WP_143049089.1,46,26,305,79,90,72,154,115,44,...,196,89,63,71,102,60,67,100,47,93


In [12]:
DicoTable = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/data/RNA_seq_DE_result/WT_vs_Ecol_ratio1_3.csv", sep = ";")
DicoTable

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Gene_name,GeneID,orgdb_symbol,orgdb_old_MXAN,orgdb_Name
0,WP_002614080.1,"17681,5595400946","1,13148580809734","0,333820387052008","3,38950481152327","0,000700189800953942","0,00401749202195971",MXAN_RS25275,41362483.0,MXAN_RS25275,MXAN_5201,30S ribosomal protein S21
1,WP_002614803.1,"13055,8658450867","-0,279785484378841","0,341134653438939","-0,820161427630867","0,41212408777437","0,55634844891773",MXAN_RS16100,41360674.0,infA,MXAN_3321,translation initiation factor IF-1
2,WP_002633201.1,"520,781350902195","-0,488659406109829","0,310280848578043","-1,57489386905206","0,115280944069584","0,223684953832229",MXAN_RS31645,41363738.0,MXAN_RS31645,MXAN_6531,HPr family phosphocarrier protein
3,WP_002633598.1,"17667,1430239249","1,78674248251241","0,432318538781286","4,1329305181991","3,58167027156953e-05","0,000318057858780743",MXAN_RS16040,41360662.0,MXAN_RS16040,MXAN_3309,50S ribosomal protein L14
4,WP_002633601.1,"4426,40026145906","0,8104626466811","0,545743984534573","1,48506015576569","0,13752788957159","0,253668278115051",MXAN_RS16025,41360659.0,MXAN_RS16025,MXAN_3306,50S ribosomal protein L16
...,...,...,...,...,...,...,...,...,...,...,...,...
6886,WP_141276995.1,"55,5707618478865","2,38043768070863","0,546005542238998","4,35973171801004","1,30221996444244e-05","0,000133398790047742",NaN,NaN,NaN,NaN,NaN
6887,WP_141277062.1,"25,2414912393685","1,38263010290852","0,762769321629563","1,81264513884054","0,0698865710503559","0,155014402646007",NaN,NaN,NaN,NaN,NaN
6888,WP_143049088.1,"2729,23193040998","0,100695081633711","0,293071005956888","0,343585955577344","0,7311576882233","0,819386506675355",MXAN_RS19445,41361331.0,NaN,NaN,NaN
6889,WP_143049089.1,"109,891161452436","2,01788957892024","0,643239069754101","3,1370755816988","0,0017064213158805","0,008435401210712",NaN,NaN,NaN,NaN,NaN


In [21]:
Dico = {}
for i in DicoTable.index:
    Dico[DicoTable.gene[i]] = DicoTable.orgdb_old_MXAN[i]

print(Dico)

{'WP_002614080.1': 'MXAN_5201', 'WP_002614803.1': 'MXAN_3321', 'WP_002633201.1': 'MXAN_6531', 'WP_002633598.1': 'MXAN_3309', 'WP_002633601.1': 'MXAN_3306', 'WP_002633602.1': 'MXAN_3305', 'WP_002633603.1': 'MXAN_3304', 'WP_002633604.1': 'MXAN_3303', 'WP_002633606.1': 'MXAN_3301', 'WP_002633607.1': 'MXAN_3300', 'WP_002633608.1': 'MXAN_3299', 'WP_002634092.1': 'MXAN_5125', 'WP_002634160.1': 'MXAN_5074', 'WP_002634235.1': 'MXAN_5002', 'WP_002634367.1': 'MXAN_5592', 'WP_002634498.1': 'MXAN_5688', 'WP_002634858.1': 'MXAN_2709', 'WP_002635080.1': 'MXAN_2913', 'WP_002635502.1': 'MXAN_1434', 'WP_002635980.1': 'MXAN_0672', 'WP_002636238.1': 'MXAN_0403', 'WP_002636478.1': 'MXAN_4033', 'WP_002636551.1': 'MXAN_4095', 'WP_002636698.1': 'MXAN_4636', 'WP_002636699.1': 'MXAN_4637', 'WP_002636736.1': 'MXAN_4673', 'WP_002637061.1': 'MXAN_2448', 'WP_002637270.1': 'MXAN_1926', 'WP_002637381.1': 'MXAN_3295', 'WP_002637840.1': 'MXAN_7512', 'WP_002638649.1': 'No old locus tag', 'WP_002639025.1': 'MXAN_2528', 

In [9]:
for i in DataTable.GeneNames:
    print(i)

WP_002614080.1
WP_002614803.1
WP_002633201.1
WP_002633598.1
WP_002633601.1
WP_002633602.1
WP_002633603.1
WP_002633604.1
WP_002633606.1
WP_002633607.1
WP_002633608.1
WP_002634092.1
WP_002634160.1
WP_002634235.1
WP_002634367.1
WP_002634498.1
WP_002634858.1
WP_002635080.1
WP_002635502.1
WP_002635980.1
WP_002636238.1
WP_002636478.1
WP_002636551.1
WP_002636698.1
WP_002636699.1
WP_002636736.1
WP_002637061.1
WP_002637270.1
WP_002637381.1
WP_002637840.1
WP_002638649.1
WP_002639025.1
WP_002639318.1
WP_002639616.1
WP_002639669.1
WP_002639727.1
WP_002640434.1
WP_002640507.1
WP_002640579.1
WP_002640911.1
WP_002640925.1
WP_011550153.1
WP_011550154.1
WP_011550155.1
WP_011550156.1
WP_011550157.1
WP_011550158.1
WP_011550159.1
WP_011550160.1
WP_011550161.1
WP_011550162.1
WP_011550163.1
WP_011550164.1
WP_011550165.1
WP_011550166.1
WP_011550168.1
WP_011550169.1
WP_011550171.1
WP_011550173.1
WP_011550174.1
WP_011550175.1
WP_011550176.1
WP_011550178.1
WP_011550179.1
WP_011550180.1
WP_011550181.1
WP_0115501

In [11]:
M_xanthus.genes.MAXN_4565

Gene identifier,MAXN_4565
Name,G_MAXN_4565
Memory address,0x7a6fc1ab0650
Functional,True
In 1 reaction(s),rxn00899_c
